**Make sure you load the API keys for cloud providers!**

You can set your environment keys yourself or use a script. Please note that since keys are private, they are not included in the repository.

In [ ]:
# setting the environment variables, the keys
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

from config import set_environment
# for the keys - as explained early in chapter 2
set_environment()

# Query Expansion

In [4]:
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

expansion_template = """Given the user question: {question}
Generate three alternative versions that express the same information need but with different wording
Make sure you dont deviate away from the information provided from the original query:
1."""

expansion_prompt = PromptTemplate(
    input_variables=["question"],
    template=expansion_template,
)

llm = ChatOpenAI(temperature=0.7)
expansion_chain = expansion_prompt | llm | StrOutputParser()

# Generate expanded queries

original_query = "What are the effects of climate change?"
expanded_queries = expansion_chain.invoke(original_query)

print(expanded_queries)

What impacts does climate change have on the environment?
2. How does climate change affect the planet?
3. What consequences result from climate change?


# Hypothetical Document Embeddings (HyDE)

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(
    file_path="knowledge_base.json",
    jq_schema=".[].content",
    text_content=True,
)

documents = loader.load()
embedder = OpenAIEmbeddings()
embeddings = embedder.embed_documents([doc.page_content for doc in documents])
vector_db = FAISS.from_documents(documents, embedder)

Hyde prompt to generate a plausible answer which is then embedded for semantic searching. 

In [ ]:
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Create Prompt for Generating Hypothetical Document
hyde_template = """Based on the question: {question}:
Write a passage that could contain the answer to this question:"""

hyde_prompt = PromptTemplate(
    input_variables=["question"],
    template=hyde_template,
)

llm = ChatOpenAI(temperature=0.3)

hyde_chain = hyde_prompt | llm | StrOutputParser()

# Generate hypothetical document
query = "What dietary changes can reduce carbon footprint?"
hypothetical_doc = hyde_chain.invoke(query)

# Use the hypothetical document for retrieval
embeddings = OpenAIEmbeddings()
embedded_query = embeddings.embed_query(hypothetical_doc)

results = vector_db.similarity_search_by_vector(embedded_query, k=3)

In [10]:
results_no_hyde = vector_db.similarity_search(query, k=3)

In [11]:
print("Results without HyDE:")
for result in results_no_hyde:
    print(result.page_content)

print("\n")

print("Results with HyDE:")
for result in results:
    print(result.page_content)

Results without HyDE:
Eating plant based alternatives have shown to reduce carbon footprint by 70% compared to eating beef.
Transformer models were introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017. The architecture relies on self-attention mechanisms rather than recurrent or convolutional neural networks. This design allows for more parallelization during training and better handling of long-range dependencies in text.
Prompt engineering involves designing and refining prompts to elicit desired responses from language models. Effective prompts can significantly improve the quality of generated text. Techniques include zero-shot prompting, few-shot prompting, and chain-of-thought prompting, where the model is guided through a series of reasoning steps.


Results with HyDE:
Eating plant based alternatives have shown to reduce carbon footprint by 70% compared to eating beef.
Prompt engineering involves designing and refining prompts to elicit desired responses

In [12]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableAssign

retrieval_chain = (
    hyde_prompt
    | llm
    | StrOutputParser()
    | RunnableLambda(lambda text: embeddings.embed_query(text))
    | RunnableLambda(lambda hyde: vector_db.similarity_search_by_vector(hyde, k=3))
)

In [14]:
from typing import Any
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableAssign

AGENT_RESPONSE_TEMPLATE = """You are an expert assistant.
Based *only* on the following retrieved documents (ignore any outside knowledge):
---
{rag_results}
---
Provide a comprehensive and concise answer to the original question: "{question}"

Only answer based on the source and keep it relevant to the question. If the source is not useful, then say you don't know.
"""

improved_response_prompt = PromptTemplate(
   input_variables=["rag_results", "question"],
   template=AGENT_RESPONSE_TEMPLATE,
)

def format_docs(docs: list[Any]) -> str:
    return "\n--\n".join([doc.page_content for doc in docs])

response_chain = (
    # Pass through the retrieval chain (previous HyDE + Retrieval)
    RunnablePassthrough.assign(rag_results=retrieval_chain)
    | {
        "rag_results": RunnableLambda(lambda docs: format_docs(docs["rag_results"])),
        "question": RunnablePassthrough(),
    }
    | improved_response_prompt
    | llm
    | StrOutputParser()
)

query = "What dietary changes can reduce carbon footprint?"

print(f"Original Query: {query}\n")
print("-" * 50)

final_answer = response_chain.invoke({"question": query})

print("\n--- Final Generated Answer ---")
print(final_answer)

Original Query: What dietary changes can reduce carbon footprint?

--------------------------------------------------

--- Final Generated Answer ---
Eating plant-based alternatives has been shown to reduce carbon footprint by 70% compared to eating beef. This dietary change can significantly lower the environmental impact of food consumption.


# Contextual Compression

In [21]:
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain.retrievers import ContextualCompressionRetriever
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0)
compressor = LLMChainExtractor.from_llm(llm)

# Create a basic retriever from the vector store
base_retriever = vector_db.as_retriever(search_kwargs={"k": 3})

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,
)

compressed_docs = compression_retriever.invoke("How do transformers work?")

In [23]:
for doc in compressed_docs:
    print(doc.page_content)
    print(doc.metadata)
    print()

Transformer models were introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017. The architecture relies on self-attention mechanisms rather than recurrent or convolutional neural networks. This design allows for more parallelization during training and better handling of long-range dependencies in text.
{'source': '/Users/lesliepan/Documents/CAMICE/generative_ai_with_langchain/chapter4/knowledge_base.json', 'seq_num': 1}

BERT (Bidirectional Encoder Representations from Transformers) was developed by Google AI Language team in 2018.
{'source': '/Users/lesliepan/Documents/CAMICE/generative_ai_with_langchain/chapter4/knowledge_base.json', 'seq_num': 2}



In [16]:
print(compressed_docs)

[Document(metadata={'source': '/Users/lesliepan/Documents/CAMICE/generative_ai_with_langchain/chapter4/knowledge_base.json', 'seq_num': 1}, page_content="Transformer models were introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017. The architecture relies on self-attention mechanisms rather than recurrent or convolutional neural networks. This design allows for more parallelization during training and better handling of long-range dependencies in text."), Document(metadata={'source': '/Users/lesliepan/Documents/CAMICE/generative_ai_with_langchain/chapter4/knowledge_base.json', 'seq_num': 3}, page_content='transformer-based neural networks'), Document(metadata={'source': '/Users/lesliepan/Documents/CAMICE/generative_ai_with_langchain/chapter4/knowledge_base.json', 'seq_num': 2}, page_content='BERT (Bidirectional Encoder Representations from Transformers) was developed by Google AI Language team in 2018.')]


In [25]:
contextual_compression_response_chain = RunnablePassthrough.assign(
    compression_retrival_results=compression_retriever
)

response_contextual_chain = (
    # Pass through the retrieval chain (previous contextual + Retrieval)
    RunnablePassthrough.assign(rag_results=contextual_compression_response_chain)
    | {
        "rag_results": RunnableLambda(lambda docs: format_docs(docs["rag_results"])),
        "question": RunnablePassthrough(),
    }
    | improved_response_prompt
    | llm
    | StrOutputParser()
)

query = "How do transformers work?"

print(f"Original Query: {query}\n")
print("-" * 50)

final_answer = response_chain.invoke({"question": query})

print("\n--- Final Generated Answer ---")
print(final_answer)

Original Query: How do transformers work?

--------------------------------------------------

--- Final Generated Answer ---
Transformers work by utilizing self-attention mechanisms instead of recurrent or convolutional neural networks. This design allows for more parallelization during training and better handling of long-range dependencies in text. GPT models, which are based on transformers, are unidirectional language models that predict the next token based on previous tokens. In contrast, BERT, another transformer-based model, is bidirectional and pre-trains deep representations by conditioning on both left and right context in all layers.


# Maximum Marginal Relevance (MMR)

In [32]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(documents, embeddings)

mmr_results = vector_db.max_marginal_relevance_search(
    query="What are transformer models?",
    k=5, # Number of documents to return
    fetch_k=20, # Number of documents to initially fetch
    lambda_mult=0.8, # Diversity parameter (0=max diversity, 1=max relevance)
)

for result in mmr_results:
    print(result.page_content)
    print()
# print(mmr_results)

Transformer models were introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017. The architecture relies on self-attention mechanisms rather than recurrent or convolutional neural networks. This design allows for more parallelization during training and better handling of long-range dependencies in text.

GPT (Generative Pre-trained Transformer) models are autoregressive language models that use transformer-based neural networks. Unlike BERT, which is bidirectional, GPT models are unidirectional and predict the next token based on previous tokens. The original GPT was introduced by OpenAI in 2018, followed by GPT-2 in 2019 and GPT-3 in 2020, each significantly larger than its predecessor.

BERT (Bidirectional Encoder Representations from Transformers) was developed by Google AI Language team in 2018. It is pre-trained using masked language modeling and next sentence prediction tasks. BERT is designed to pre-train deep bidirectional representations by jointly condi

# Source Attribution

In [37]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document


# Example documents
documents = [
    Document(
        page_content="The transformer architecture was introduced in the paper 'Attention is All You Need' by Vaswani et al. in 2017.",
        metadata={"source": "Neural Network Review 2021", "page": 42},
    ),
    Document(
        page_content="BERT uses bidirectional training of the Transformer, masked language modeling, and next sentence prediction tasks.",
        metadata={"source": "Introduction to NLP", "page": 137},
    ),
    Document(
        page_content="GPT models are autoregressive transformers that predict the next token based on previous tokens.",
        metadata={"source": "Large Language Models Survey", "page": 89},
    ),
]

embeddings = OpenAIEmbeddings()
vector_store = FAISS.from_documents(documents, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# Source attribution prompt template
attribution_prompt = ChatPromptTemplate.from_template("""
You are a precise AI assistant that provides well-sourced information. 
Answer the following question based ONLY on the provided sources. For each fact or claim in your answer,
include a citation useing [1], [2], etc. that refers to the source. Include a numbererd reference list at the end. 

Question: {question}

Sources:
{sources}                                               

You answer:                                              
""")

# Create a source-formatted string from documents
def format_sources_with_citations(docs):
    formatted_sources = []
    for i, doc in enumerate(docs, 1):
        source_info = f"[{i}] {doc.metadata.get('source', 'Unknown Source')}"
        if doc.metadata.get("page"):
            source_info += f", page {doc.metadata['page']}"
        formatted_sources.append(f"{source_info}\n{doc.page_content}")
    return "\n\n".join(formatted_sources)


# Build the RAG chain with source attribution
def generate_attributed_response(question):
    # Retrieve relevant documents
    retrieved_docs = retriever.invoke(question)

    # Format sources with citation numbers
    sources_formatted = format_sources_with_citations(retrieved_docs)

    # Create the attribution chain using LCEL
    attribution_chain = (
        attribution_prompt
        | ChatOpenAI(temperature=0)
        | StrOutputParser()
    )

    response = attribution_chain.invoke({
        "question": question,
        "sources": sources_formatted,
    })

    return response

In [39]:
# Example Usage
question = "How do transformer models work and what are some examples?"
attributed_answer = generate_attributed_response(question)
print(attributed_answer)

Transformer models work by utilizing self-attention mechanisms to weigh the importance of different input tokens when making predictions. This allows them to capture long-range dependencies in data more effectively compared to traditional recurrent neural networks. One of the key features of transformer models is their ability to process input data in parallel, making them more efficient for training and inference tasks [1].

Some examples of transformer models include BERT and GPT. BERT, introduced by Google in 2018, utilizes bidirectional training of the Transformer, masked language modeling, and next sentence prediction tasks to achieve state-of-the-art performance on various natural language processing tasks [2]. On the other hand, GPT models, developed by OpenAI, are autoregressive transformers that predict the next token in a sequence based on previous tokens. This allows them to generate coherent and contextually relevant text, making them suitable for tasks like text generation

# Self-consistency Checking

In [40]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document


def verify_response_accuracy(
    retrieved_docs: list[Document],
    generated_answer: str,
    llm: ChatOpenAI = None,
) -> dict:
    """Verifies if a generated answer is fully supported by the retrieved documents.

    Args:
        retrieved_docs (list[Document]): List of documents used to generate the answer
        generated_answer (str): The answer produced by the RAG system
        llm (ChatOpenAI, optional): LLM to use for verfication. Defaults to None.

    Returns:
        dict: Dictionary containing verification results and any identified issues
    """
    if llm is None:
        llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

    # Create context from retrieved documents
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    # Define verification prompt - fixed to avoid JSON formatting issues in the template
    verification_prompt = ChatPromptTemplate.from_template(
        """
    As a fact-checking assistant, verify whether the following answer is fully supported
    by the provided context. Identify any statements that are not supported or contradict the context.

    Context:
    {context}
                                                           
    Answer to verify:
    {answer}
                                                           
    Perform a detailed analysis with the following structure:
    1. List any factual claims in the answer
    2. For each claim, indicate whether it is:
        - Fully supperted )provide the supporting text from the context)
        - Partially supported (explain what parts lack support)
        - Contradicted (identify the contradiction)
        - Not mentioned in the context
    3. Overall assessment: Is the answer fully grounded in the context?
                                                           
    Return your analysis in JSON format with the following structure:
    {{
        "claims": [
            {{
                "claim": "The factual claim",
                "status": "fully_supported|partially_supported|contradicted|not_mentioned",
                "evidence": "Supporting of contradicting text from context",
                "explanation": "You explanation", 
            }}
        ],
        "fully_grounded": true|false,
        "issues_identified": ["List any specific issues"]
    }}
                                                           
    """
    )

    verification_chain = verification_prompt | llm | StrOutputParser()

    # Run Verification
    result = verification_chain.invoke(
        {
            "context": context,
            "answer": generated_answer,
        }
    )

    return result


# Example usage
retrieved_docs = [
    Document(
        page_content="The transformer architecture was introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017. It relies on self-attention mechanisms instead of recurrent or convolutional neural networks."
    ),
    Document(
        page_content="BERT is a transformer-based model developed by Google that uses masked language modeling and next sentence prediction as pre-training objectives."
    ),
]

generated_answer = "The transformer architecture was introduced by OpenAI in 2018 and uses recurrent neural networks. BERT is a transformer model developed by Google."

verification_result = verify_response_accuracy(retrieved_docs, generated_answer)
print(verification_result)

{
    "claims": [
        {
            "claim": "The transformer architecture was introduced by OpenAI in 2018",
            "status": "contradicted",
            "evidence": "The transformer architecture was introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017.",
            "explanation": "The claim is contradicted by the fact that the transformer architecture was introduced in 2017 by Vaswani et al., not by OpenAI in 2018."
        },
        {
            "claim": "The transformer architecture uses recurrent neural networks",
            "status": "contradicted",
            "evidence": "It relies on self-attention mechanisms instead of recurrent or convolutional neural networks.",
            "explanation": "The claim is contradicted by the fact that the transformer architecture does not use recurrent neural networks, but self-attention mechanisms."
        },
        {
            "claim": "BERT is a transformer model developed by Google",
            "s